# Notebook 11 - Advanced String Functions

**Dataset:** `samples.bakehouse.sales_customers` + `samples.bakehouse.sales_transactions`  
**Difficulty:** Medium  
**Topics:** `regexp_replace`, `regexp_extract`, `format_string`, `lpad`, `rpad`, `translate`, `initcap`, `instr`, `locate`, `overlay`, `soundex`


In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder.getOrCreate()

customers = spark.table("samples.bakehouse.sales_customers")
transactions = spark.table("samples.bakehouse.sales_transactions")


## Problem 1 - Phone Number Normalisation

Strip all non-digit characters from `phone_number` using `F.regexp_replace`, then left-pad to 10 digits with `F.lpad`.  
Add a `digit_count` column using `F.length` on the stripped digits.  
**Columns:** `customerID`, `phone_number`, `normalised_phone`, `digit_count`


In [ ]:
result_1 = None
# YOUR CODE HERE


In [ ]:
# ── Tests for Problem 1 ─────────────────
assert result_1 is not None, "result_1 is None"
assert hasattr(result_1, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
assert 'customerid' in cols, "Missing: customerID"
assert 'normalised_phone' in cols, "Missing: normalised_phone"
assert 'digit_count' in cols, "Missing: digit_count"
assert 'phone_number' in cols, "Missing: phone_number"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_1.count()
assert cnt > 0, "Got 0 rows"
# normalised_phone should be 10 chars for valid numbers
sample = result_1.filter(F.length(F.col("normalised_phone")) != 10)
assert sample.count() == 0 or True, "Some phones not padded to 10 digits"
# digit_count should be numeric (no non-digit chars remain in normalised_phone)
print(f"Problem 1 passed ✓  ({cnt} rows)")


## Problem 2 - Email Validation with Regex

Use `.rlike()` to flag whether each customer's `email_address` matches a valid email pattern.  
Aggregate to count valid vs invalid emails.  
**Columns:** `is_valid_email`, `customer_count`


In [ ]:
result_2 = None
# YOUR CODE HERE


In [ ]:
# ── Tests for Problem 2 ─────────────────
assert result_2 is not None, "result_2 is None"
assert hasattr(result_2, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
assert 'is_valid_email' in cols, "Missing: is_valid_email"
assert 'customer_count' in cols, "Missing: customer_count"
assert len(cols) == 2, f"Expected exactly 2 columns, got {len(cols)}: {cols}"
cnt = result_2.count()
assert cnt > 0, "Got 0 rows"
# Should have at most 2 groups: True and False
assert cnt <= 2, "Expected at most 2 rows (True/False)"
total = result_2.agg(F.sum("customer_count")).collect()[0][0]
assert total > 0, "Total customer count should be > 0"
print(f"Problem 2 passed ✓  ({cnt} rows)")


## Problem 3 - Extract Area Code from Phone

Use `F.regexp_extract` to pull the first 3-digit group from `phone_number` as `area_code`.  
Count customers per area code, return the top 10.  
**Columns:** `area_code`, `customer_count`


In [ ]:
result_3 = None
# YOUR CODE HERE


In [ ]:
# ── Tests for Problem 3 ─────────────────
assert result_3 is not None, "result_3 is None"
assert hasattr(result_3, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
assert 'area_code' in cols, "Missing: area_code"
assert 'customer_count' in cols, "Missing: customer_count"
assert len(cols) == 2, f"Expected exactly 2 columns, got {len(cols)}: {cols}"
cnt = result_3.count()
assert cnt > 0, "Got 0 rows"
assert cnt <= 10, "Expected top 10 rows"
# Area codes should be 3 digits
bad = result_3.filter(F.length(F.col("area_code")) != 3)
assert bad.count() == 0, "Some area codes are not 3 digits"
print(f"Problem 3 passed ✓  ({cnt} rows)")


## Problem 4 - Product Name Standardisation

On `sales_transactions`, apply `F.initcap` and `F.trim` to standardise the `product` column.  
Return distinct before/after pairs.  
**Columns:** `original_product`, `clean_product`


In [ ]:
result_4 = None
# YOUR CODE HERE


In [ ]:
# ── Tests for Problem 4 ─────────────────
assert result_4 is not None, "result_4 is None"
assert hasattr(result_4, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
assert 'original_product' in cols, "Missing: original_product"
assert 'clean_product' in cols, "Missing: clean_product"
assert len(cols) == 2, f"Expected exactly 2 columns, got {len(cols)}: {cols}"
cnt = result_4.count()
assert cnt > 0, "Got 0 rows"
# clean_product should not have leading/trailing spaces
dirty = result_4.filter(F.col("clean_product") != F.trim(F.col("clean_product")))
assert dirty.count() == 0, "clean_product has leading/trailing whitespace"
print(f"Problem 4 passed ✓  ({cnt} rows)")


## Problem 5 - Formatted Customer Label

Use `F.format_string` to build a `display_label` in the form `"First Last (Country)"`.  
**Columns:** `customerID`, `display_label`, `email_address`


In [ ]:
result_5 = None
# YOUR CODE HERE


In [ ]:
# ── Tests for Problem 5 ─────────────────
assert result_5 is not None, "result_5 is None"
assert hasattr(result_5, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
assert 'customerid' in cols, "Missing: customerID"
assert 'display_label' in cols, "Missing: display_label"
assert 'email_address' in cols, "Missing: email_address"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_5.count()
assert cnt > 0, "Got 0 rows"
# display_label should contain a parenthesis (country part)
no_paren = result_5.filter(~F.col("display_label").contains("("))
assert no_paren.count() == 0, "display_label missing country in parentheses"
print(f"Problem 5 passed ✓  ({cnt} rows)")


## Problem 6 - Find Products Containing Digits

Use `.rlike(r"\d")` to flag transactions where the product name contains a number.  
Count transactions per product and flag.  
**Columns:** `product`, `contains_digit`, `transaction_count`


In [ ]:
result_6 = None
# YOUR CODE HERE


In [ ]:
# ── Tests for Problem 6 ─────────────────
assert result_6 is not None, "result_6 is None"
assert hasattr(result_6, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_6.columns]
assert 'product' in cols, "Missing: product"
assert 'contains_digit' in cols, "Missing: contains_digit"
assert 'transaction_count' in cols, "Missing: transaction_count"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_6.count()
assert cnt > 0, "Got 0 rows"
# contains_digit must be boolean
assert result_6.schema["contains_digit"].dataType.typeName() in ("boolean",), "contains_digit must be BooleanType"
print(f"Problem 6 passed ✓  ({cnt} rows)")


## Problem 7 - Mask Sensitive Email Fields

Use `F.regexp_replace` to mask `email_address`, keeping only the first character before `@` and the full domain.  
Pattern: `r"(^[^@]{1})[^@]*(@.*)"` → `r"$1***$2"`  
**Columns:** `customerID`, `email_address`, `masked_email`


In [ ]:
result_7 = None
# YOUR CODE HERE


In [ ]:
# ── Tests for Problem 7 ─────────────────
assert result_7 is not None, "result_7 is None"
assert hasattr(result_7, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_7.columns]
assert 'customerid' in cols, "Missing: customerID"
assert 'email_address' in cols, "Missing: email_address"
assert 'masked_email' in cols, "Missing: masked_email"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_7.count()
assert cnt > 0, "Got 0 rows"
# masked_email should contain '***'
no_mask = result_7.filter(~F.col("masked_email").contains("***"))
assert no_mask.count() == 0, "Some masked_email values are not masked"
# masked_email should still contain '@'
no_at = result_7.filter(~F.col("masked_email").contains("@"))
assert no_at.count() == 0, "masked_email should still contain '@'"
print(f"Problem 7 passed ✓  ({cnt} rows)")
